In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import csv

In [5]:
def focused_crawl(url, keyword, depth = 1, visited = None, csv_writer=None,parent_url=None):
    if visited is None:
        visited = set()
    if depth == 0 or url in visited:
        return

    print(f"Crawling: {url} (Found on: {parent_url})")
    visited.add(url)
    try:
        response = requests.get(url, timeout=5)
        if response.status_code != 200:
            return
        soup = BeautifulSoup(response.text,'html.parser')
        if keyword.lower() in soup.get_text().lower():
            print(f"Keyword '{keyword}' found on {url} ")
            if csv_writer:
                csv_writer.writerow([url, parent_url])
        links = [urljoin(url,a["href"]) for a in soup.find_all("a", href = True)]
        for link in links:
            if link.startswith("http") or link.startswith("https"):
                focused_crawl(link,keyword,depth-1,visited, csv_writer,url)
    except requests.RequestException as e:
        print(f"Error: {e}")

start_url = "https://www.scrapingbee.com/blog/crawling-python/"
search_keyword = "scraping"

with open("focused_crawled_links.csv", "w", newline = "") as file:
    writer = csv.writer(file)
    writer.writerow(["URL","Found On"])
    focused_crawl(start_url, search_keyword, depth=2, csv_writer = writer, parent_url="Root")

    

Crawling: https://www.scrapingbee.com/blog/crawling-python/ (Found on: Root)
Keyword 'scraping' found on https://www.scrapingbee.com/blog/crawling-python/ 
Crawling: https://dashboard.scrapingbee.com/request-builder-fast-search (Found on: https://www.scrapingbee.com/blog/crawling-python/)
Error: HTTPSConnectionPool(host='dashboard.scrapingbee.com', port=443): Max retries exceeded with url: /request-builder-fast-search (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000016787F78A50>, 'Connection to dashboard.scrapingbee.com timed out. (connect timeout=5)'))
Crawling: https://www.scrapingbee.com/ (Found on: https://www.scrapingbee.com/blog/crawling-python/)
Keyword 'scraping' found on https://www.scrapingbee.com/ 
Crawling: https://dashboard.scrapingbee.com/account/login (Found on: https://www.scrapingbee.com/blog/crawling-python/)
Error: HTTPSConnectionPool(host='dashboard.scrapingbee.com', port=443): Max retries exceeded with url: /account/login (Caused 